In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jaiahuja/skin-cancer-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skin-cancer-detection' dataset.
Path to dataset files: /kaggle/input/skin-cancer-detection


In [5]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D , MaxPooling2D ,Flatten , Dense , Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.05,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

In [7]:
training_generator = datagen.flow_from_directory(
    "/kaggle/input/skin-cancer-detection/Skin cancer ISIC The International Skin Imaging Collaboration/Train",
    target_size = (48,48),
    batch_size = 64,
    shuffle=True,
    color_mode="rgb",
    class_mode="categorical",
    subset="training"
)

Found 1795 images belonging to 9 classes.


In [8]:
val_generator = datagen.flow_from_directory(
    "/kaggle/input/skin-cancer-detection/Skin cancer ISIC The International Skin Imaging Collaboration/Train",
    target_size = (48,48),
    batch_size = 64,
    shuffle=True,
    color_mode="rgb",
    class_mode="categorical",
    subset = 'validation'
)

Found 444 images belonging to 9 classes.


In [9]:
test_datagen = ImageDataGenerator(rescale=1./255)

In [10]:
test_generator = test_datagen.flow_from_directory(
    "/kaggle/input/skin-cancer-detection/Skin cancer ISIC The International Skin Imaging Collaboration/Test",
    target_size = (48,48),
    batch_size=64,
    shuffle=False,
    color_mode="rgb",
    class_mode="categorical"
)

Found 118 images belonging to 9 classes.


In [12]:
model = Sequential([
    Conv2D(64,(3,3), activation='relu', input_shape=(48,48,3)),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(128,(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    Conv2D(128,(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    Conv2D(256,(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.25),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.25),

    Dense(64, activation='relu'),
    Dropout(0.25),

    Dense(32, activation='relu'),

    Dense(9, activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile (
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ['accuracy']
)

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 2, 2, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 1, 1, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 1, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 9)              │           297 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 561,929 (2.14 MB)

 Trainable params: 561,929 (2.14 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    training_generator,
    validation_data = val_generator,
    epochs = 100,
)

Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 24s 815ms/step - accuracy: 0.1994 - loss: 2.0499 - val_accuracy: 0.1959 - val_loss: 2.0142
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 712ms/step - accuracy: 0.2357 - loss: 1.9606 - val_accuracy: 0.2500 - val_loss: 1.9154
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 726ms/step - accuracy: 0.2858 - loss: 1.8965 - val_accuracy: 0.2883 - val_loss: 1.9858
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 726ms/step - accuracy: 0.2869 - loss: 1.8554 - val_accuracy: 0.3288 - val_loss: 1.7532
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 719ms/step - accuracy: 0.3437 - loss: 1.7519 - val_accuracy: 0.3243 - val_loss: 1.7742
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 723ms/step - accuracy: 0.3415 - loss: 1.7297 - val_accuracy: 0.3041 - val_loss: 1.7640
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 22s 747ms/step - accuracy: 0.3577 - loss: 1.6926 - val_accuracy: 0.2995 - val_loss: 1.7502
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 722ms/step - accuracy: 0.3643 - loss: 1.6765 - 

In [19]:
model.save("skin_cancer_cnn.h5")

In [18]:
import json

class_names = [
    name for name, index in sorted(
        training_generator.class_indices.items(),
        key=lambda item: item[1]
    )
]

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print(class_names)

['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
